In [1]:
from pyspark.sql import SparkSession
import time
import pandas as pd
from pyspark.sql.functions import min, max, mean, stddev


## Q1.

In [2]:
spark = (
    SparkSession.builder
    .appName("TradeCorpETL")
    .getOrCreate()
)

print("Version de Spark :", spark.version)

Version de Spark : 4.2.0


## Q2.

In [17]:
base_path = "/home/jovyan/data/raw"

df_categories = spark.read.csv(
    f"{base_path}/categories.csv",
    header=True,
    inferSchema=True
)

df_customers = spark.read.csv(
    f"{base_path}/customers.csv",
    header=True,
    inferSchema=True
)

df_employees = spark.read.csv(
    f"{base_path}/employees.csv",
    header=True,
    inferSchema=True
)

df_order_details = spark.read.csv(
    f"{base_path}/order_details.csv",
    header=True,
    inferSchema=True
)

df_orders = spark.read.csv(
    f"{base_path}/orders.csv",
    header=True,
    inferSchema=True
)

df_products = spark.read.csv(
    f"{base_path}/products.csv",
    header=True,
    inferSchema=True
)

df_shippers = spark.read.csv(
    f"{base_path}/shippers.csv",
    header=True,
    inferSchema=True
)

df_suppliers = spark.read.csv(
    f"{base_path}/suppliers.csv",
    header=True,
    inferSchema=True
)

## Q3.

In [18]:
dataframes = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers
}

for name, df in dataframes.items():
    print(f"\n===== {name.upper()} =====")
    df.printSchema()


===== CATEGORIES =====
root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)


===== CUSTOMERS =====
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- fax: string (nullable = true)


===== EMPLOYEES =====
root
 |-- employee_id: integer (nullable = true)
 |-- last_name: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- title_of_courtesy: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- ad

## Q4.

In [19]:
for name, df in dataframes.items():
    print(f"\n===== {name.upper()} =====")
    df.show(5)


===== CATEGORIES =====
+-----------+--------------+--------------------+-------+
|category_id| category_name|         description|picture|
+-----------+--------------+--------------------+-------+
|          1|     Beverages|Soft drinks, coff...|   NULL|
|          2|    Condiments|Sweet and savory ...|   NULL|
|          3|   Confections|Desserts, candies...|   NULL|
|          4|Dairy Products|             Cheeses|   NULL|
|          5|Grains/Cereals|Breads, crackers,...|   NULL|
+-----------+--------------+--------------------+-------+
only showing top 5 rows

===== CUSTOMERS =====
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|       city|region|postal_code|country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+---

## Q5.

In [20]:
counts = []

for name, df in dataframes.items():
    counts.append((name, df.count()))

counts_df = spark.createDataFrame(
    counts,
    ["DataFrame", "Nombre_de_lignes"]
)

counts_df.show()

+-------------+----------------+
|    DataFrame|Nombre_de_lignes|
+-------------+----------------+
|   categories|               8|
|    customers|              91|
|    employees|               9|
|order_details|            2155|
|       orders|             830|
|     products|              77|
|     shippers|               6|
|    suppliers|              29|
+-------------+----------------+



## Q6.

In [24]:
orders_df.select(
    min("freight").alias("min_freight"),
    max("freight").alias("max_freight"),
    mean("freight").alias("mean_freight"),
    stddev("freight").alias("stddev_freight")
).show()

+-----------+-----------+-----------------+------------------+
|min_freight|max_freight|     mean_freight|    stddev_freight|
+-----------+-----------+-----------------+------------------+
|       0.02|    1007.64|78.24420481927719|116.77929363024192|
+-----------+-----------+-----------------+------------------+



In [26]:
products_df.select(
    min("unit_price").alias("min_unit_price"),
    max("unit_price").alias("max_unit_price"),
    mean("unit_price").alias("mean_unit_price"),
    stddev("unit_price").alias("stddev_unit_price"),
).show()
products_df.select(
    min("units_in_stock").alias("min_units_in_stock"),
    max("units_in_stock").alias("max_units_in_stock"),
    mean("units_in_stock").alias("mean_units_in_stock"),
    stddev("units_in_stock").alias("stddev_units_in_stock")
).show()


+--------------+--------------+------------------+-----------------+
|min_unit_price|max_unit_price|   mean_unit_price|stddev_unit_price|
+--------------+--------------+------------------+-----------------+
|           2.5|         263.5|28.833896103896105|33.82931122234885|
+--------------+--------------+------------------+-----------------+

+------------------+------------------+-------------------+---------------------+
|min_units_in_stock|max_units_in_stock|mean_units_in_stock|stddev_units_in_stock|
+------------------+------------------+-------------------+---------------------+
|                 0|               125| 40.506493506493506|    36.14722213124929|
+------------------+------------------+-------------------+---------------------+



## Q7.

La Lazy Evaluation signifie que Spark n’exécute pas immédiatement les transformations. Il construit d’abord un plan d’exécution (DAG), puis lance réellement les calculs lorsqu’une action est appelée.

Transformations : créent un nouveau DataFrame sans exécuter immédiatement le calcul.
Ex : select(), filter(), groupBy().
Actions : déclenchent réellement l’exécution.
Ex : show(), count(), collect().

## Q8.

- Job : ensemble de calculs déclenché par une action comme show() ou count().
- Stage : sous-partie d’un job, découpée selon les étapes du traitement.
- Task : Chaque Stage est lui-même découpé en Tasks, plus petite unité de travail exécutée sur une partition de données.

## Q9. 

In [28]:
start = time.perf_counter()

orders_pandas = pd.read_csv("/home/jovyan/data/raw/orders.csv")

end = time.perf_counter()

pandas_time = end - start

print(f"Temps de lecture Pandas : {pandas_time:.6f} secondes")

Temps de lecture Pandas : 0.031580 secondes


In [32]:
start = time.perf_counter()

orders_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/home/jovyan/data/raw/orders.csv")
)

orders_spark.count()

end = time.perf_counter()

spark_time = end - start

print(f"Temps de lecture Spark : {spark_time:.6f} secondes")

Temps de lecture Spark : 0.620909 secondes


In [33]:
print(f"Pandas : {pandas_time:.6f} secondes")
print(f"Spark  : {spark_time:.6f} secondes")

difference = spark_time - pandas_time

print(f"Différence : {difference:.6f} secondes")


Pandas : 0.031580 secondes
Spark  : 0.620909 secondes
Différence : 0.589329 secondes


Pour le fichier `orders.csv`, Pandas est plus rapide que Spark car le jeu de données est de petite taille.

## Q10.


In [37]:
orders_df.columns

['order_id',
 'customer_id',
 'employee_id',
 'order_date',
 'required_date',
 'shipped_date',
 'ship_via',
 'freight',
 'ship_name',
 'ship_address',
 'ship_city',
 'ship_region',
 'ship_postal_code',
 'ship_country']

In [36]:
orders_df.dtypes

[('order_id', 'int'),
 ('customer_id', 'string'),
 ('employee_id', 'int'),
 ('order_date', 'date'),
 ('required_date', 'date'),
 ('shipped_date', 'date'),
 ('ship_via', 'int'),
 ('freight', 'double'),
 ('ship_name', 'string'),
 ('ship_address', 'string'),
 ('ship_city', 'string'),
 ('ship_region', 'string'),
 ('ship_postal_code', 'string'),
 ('ship_country', 'string')]